# 01 — Exploration des données, qualité d'image et découpage par patient

**Rôle** : comprendre les données, caler les seuils de qualité d'image, vérifier que le détecteur de type d'examen ne rejette pas de vraies radios, et figer un découpage **par patient** (dev / calibration / test).

**Sorties** : `splits.json` (à copier dans `data/splits/` du dépôt et à commiter), figures pour les slides, proposition de seuils qualité.

> Un même patient a souvent plusieurs radios. S'il apparaît dans deux splits, les résultats sont artificiellement bons. Ce notebook l'empêche.

In [ ]:
# --- Initialisation (identique dans tous les notebooks) ---
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
IN_KAGGLE = Path("/kaggle").exists()
if IN_KAGGLE:
    # Internet doit être ACTIVÉ dans les réglages du notebook Kaggle.
    # Remplacez VOTRE_COMPTE par votre identifiant GitHub (dépôt medicure-ai).
    %pip install -q git+https://github.com/VOTRE_COMPTE/medicure-ai.git torchxrayvision
else:
    sys.path.insert(0, str(Path.cwd().parent))  # notebook lancé depuis le dossier notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from medguard.config import TARGETS, MODEL_WEIGHTS, Settings, load_settings, save_settings
from medguard.paths import find_file

# --- Configuration (à adapter) ---
DATASET = os.environ.get("MEDGUARD_DATASET", "nih")            # "nih" ou "chexpert"
DATA_ROOT = Path(os.environ.get("MEDGUARD_DATA", "/kaggle/input/data"))  # À ADAPTER au dataset attaché
WEIGHTS = os.environ.get("MEDGUARD_WEIGHTS", MODEL_WEIGHTS)
N_MAX = int(os.environ.get("MEDGUARD_NMAX", 6000))               # nb max d'images chargées
SEED = 42

WORK = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("MEDGUARD_WORK", "../outputs")).resolve()
ARTIFACTS = WORK / "artifacts"
FIGURES = WORK / "figures"
for d in (ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
print("Kaggle :", IN_KAGGLE, "| dataset :", DATASET, "| poids :", WEIGHTS, "| sortie :", WORK)

## 1. Chargement

In [ ]:
from medguard.data import load_dataset

df, EVAL_TARGETS = load_dataset(DATASET, DATA_ROOT, n_max=N_MAX, seed=SEED)
print(f"{len(df)} images, {df.patient_id.nunique()} patients | cibles évaluables : {EVAL_TARGETS}")
if "Lung Opacity" not in EVAL_TARGETS:
    print("ATTENTION : ce dataset n'a pas d'étiquette 'Lung Opacity'. Cette cible ne pourra être ni calibrée ni évaluée ici.")
df.head()

## 2. Prévalence des anomalies et composition démographique

Un modèle évalué sur un échantillon déséquilibré (âge, sexe, incidence AP/PA) peut cacher des biais. On les documente dès maintenant : ce sera la diapositive « limites ».

In [ ]:
prev = pd.DataFrame({t: [df[t].sum(), df[t].mean()] for t in EVAL_TARGETS}, index=["positifs", "prévalence"]).T
display(prev)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
prev["prévalence"].plot.bar(ax=axes[0], color="#4b7bec", title="Prévalence"); axes[0].set_ylim(0, 1)
df["age"].dropna().clip(0, 100).plot.hist(bins=20, ax=axes[1], color="#8395a7", title="Âge")
pd.crosstab(df["sex"].fillna("?"), df["view"].fillna("?")).plot.bar(ax=axes[2], title="Sexe × incidence")
plt.tight_layout(); plt.savefig(FIGURES / "01_demographics.png", dpi=150); plt.show()

## 3. Distribution des métriques de qualité sur de vraies images

Les seuils par défaut de `medguard/config.py` sont des valeurs de départ. Ici on regarde ce que donnent de vraies radios pour les **caler**.

In [ ]:
from medguard.imaging import load_image
from medguard.quality import compute_metrics

sample = df.sample(min(len(df), 800), random_state=SEED)
metrics = pd.DataFrame([compute_metrics(load_image(p)) for p in sample.path])
display(metrics.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2).T)

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, col in zip(axes, ["blur_laplacian_var", "contrast_range", "brightness_dev", "overexposed_fraction"]):
    metrics[col].plot.hist(bins=30, ax=ax, color="#20bf6b"); ax.set_title(col)
plt.tight_layout(); plt.savefig(FIGURES / "01_quality_metrics.png", dpi=150); plt.show()

### Proposition de seuils

Règle simple : « mauvais » = 1er percentile (les 1 % d'images les pires sont rejetés), « bon » = 25e percentile pour les métriques où plus est mieux (75e pour celles où moins est mieux). **Regardez les histogrammes avant d'appliquer** : si votre dataset contient déjà beaucoup d'images floues, le 1er percentile sera trop tolérant.

In [ ]:
from dataclasses import asdict
from medguard.config import QualityThresholds

q = metrics.quantile([.01, .25, .75, .99])
proposal = QualityThresholds(
    blur=(float(q.loc[.01, "blur_laplacian_var"]), float(q.loc[.25, "blur_laplacian_var"])),
    contrast=(float(q.loc[.01, "contrast_range"]), float(q.loc[.25, "contrast_range"])),
    brightness_dev=(float(q.loc[.99, "brightness_dev"]), float(q.loc[.75, "brightness_dev"])),
    resolution=(128.0, 512.0),
    overexposed_fraction=(max(float(q.loc[.99, "overexposed_fraction"]), 0.35), max(float(q.loc[.75, "overexposed_fraction"]), 0.10)),
)
print(asdict(proposal))

APPLY_QUALITY_PROPOSAL = False   # passez à True après avoir regardé les histogrammes
if APPLY_QUALITY_PROPOSAL:
    settings = load_settings(ARTIFACTS / "thresholds.json")
    settings.quality = proposal
    print("Écrit :", save_settings(settings, ARTIFACTS / "thresholds.json"))

## 4. Le détecteur de type d'examen rejette-t-il de vraies radios ?

C'est une heuristique. Sur de vraies radiographies thoraciques, le taux de rejet doit être proche de 0 %. S'il ne l'est pas, ajustez `exam_min_confidence` ou entraînez un petit classifieur.

In [ ]:
from medguard.exam_check import check_exam_type

checks = [check_exam_type(load_image(p)) for p in sample.path.head(300)]
rej = np.mean([not c.is_chest_xray for c in checks])
print(f"Taux de rejet sur de vraies radios : {rej:.1%}  (cible : ~0 %)")
pd.DataFrame([c.details for c in checks]).describe().round(3)

## 5. Découpage par patient (une seule fois !)

- **dev** : régler les seuils de fiabilité (notebook 04)
- **calibration** : ajuster la calibration (notebook 03)
- **test** : rapporter les résultats finaux, jamais utilisé pour choisir quoi que ce soit

In [ ]:
from medguard.splits import group_split, save_splits

df["split"] = group_split(df.patient_id.values, {"dev": 0.20, "calibration": 0.30, "test": 0.50}, seed=SEED)

# Garde-fou : aucun patient dans deux splits
overlap = df.groupby("patient_id")["split"].nunique().max()
assert overlap == 1, "Un patient apparaît dans plusieurs splits !"

display(df.groupby("split").agg(images=("image_id", "count"), patients=("patient_id", "nunique"),
                                **{f"prev_{t}": (t, "mean") for t in EVAL_TARGETS}).round(3))

path = save_splits(WORK / "splits.json", df.image_id.values, df.split.values,
                   meta={"dataset": DATASET, "seed": SEED, "n_max": N_MAX, "group": "patient_id"})
print("Écrit :", path, "\n-> à copier dans data/splits/ du dépôt et à commiter.")

### Suite
`02_baseline_eval.ipynb` : inférence du modèle pré-entraîné et AUROC.